# Load data

In [1]:
import sys
sys.path.append('../../Common/') # Module path for CommonMT5

import CommonMT5 # File for common functions

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import talib 

def calculate_momentum(data, period=14):
	data['Momentum'] = talib.MOM(data['Close'], timeperiod=period)
	# Hoặc Momentum = Giá thị trường hiện tại – Giá trị trung bình trong một khoảng thời gian nhất định

def calculate_rsi(data, period=14):
	# Tính toán RSI
	data['RSI'] = talib.RSI(data['Close'], timeperiod=period)

def detectSignal(symbol, from_date, to_date, timeframe):

	import pandas as pd
	import plotly.graph_objects as go
	import redis
	import numpy as np
	from datetime import datetime

	# ##############################################Step 1: Lấy dữ liệu##############################################
	data = CommonMT5.CommonMT5.loaddataMT5_FromTo(symbol, from_date, to_date, timeframe)
	# data.set_index('Datetime', inplace=True)
	print(data)
	# ##############################################Step 2: Chiến lược##############################################  
	# Tính toán các chỉ báo kỹ thuật
	calculate_momentum(data)
	calculate_rsi(data)   

	# Đảm bảo không có giá trị NaN trong Momentum, RSI
	data['Momentum'] = data['Momentum'].bfill().ffill() # Điền giá trị NaN bằng phương pháp backward fill và forward fill
	data['RSI'] = data['RSI'].bfill().ffill()

	# Tạo features (X) và target (y)
	X = data[['Close', 'Momentum', 'RSI']].shift(1)  # Dùng dữ liệu của ngày hôm trước để dự đoán
	y = data['Close']  # Giá đóng cửa mà chúng ta muốn dự đoán
  
	# Tiếp tục với việc chia dữ liệu
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
	X_train = X_train.bfill().ffill()
	y_train = y_train.bfill().ffill()

	# Huấn luyện mô hình
	model = LinearRegression()
	model.fit(X_train, y_train)

	# Đảm bảo không có giá trị NaN trong X trước khi dự đoán
	X = X.bfill().ffill()

	# Dự đoán giá đóng cửa sử dụng X đã điền giá trị NaN
	data['Predicted_Close'] = model.predict(X)

	# Có thể sử dụng MSE để đánh giá mô hình ổn thì làm bước kế tiếp
	# Tính Mean Squared Error (MSE) giữa giá trị thực tế và giá trị dự đoán trên tập kiểm tra
	mse = mean_squared_error(data['Close'], data['Predicted_Close'])
	print(f'Mean Squared Error (MSE): {mse}')

	# Định nghĩa tín hiệu mua dựa trên điều kiện: Dự đoán giá tăng, Momentum dương và RSI dưới 70
	data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['Momentum'] > 0) & (data['RSI'] < 50))

	# Định nghĩa tín hiệu bán dựa trên điều kiện: Dự đoán giá giảm, Momentum âm và RSI trên 30
	data['Sell_Signal'] = ((data['Predicted_Close'] < data['Close']) & (data['Momentum'] < 0) & (data['RSI'] > 50))

	# data.to_csv("OG_FX_ML, Momentum, RSI.csv")
	print(data)
	# ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
	# Nếu có tín hiệu thì đẩy qua Redis
	# Datetime  Open    High    Low	Close   Volume  SMA short_ema   long_ema    MACD    Signal_Line     Buy_Signal      Sell_Signal
	# Tạo kết nối Redis
	r = redis.Redis(host='localhost', port=6379, db=15) # 0-5 CK, 6-10 Crypto, 11-15 Forex
	# Tạo hash key
	hash_key = 'OG_FX_ML, Momentum, RSI_K13'
	last_record = data.iloc[-1]
   
	# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
	if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
		for field, value in last_record.to_dict().items():
			# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
			if isinstance(value, pd.Timestamp):
				value = value.isoformat()
			elif isinstance(value, (int, np.uint64)):
				value = str(value)
			r.hset(hash_key, field, value)  
			r.hset(hash_key, 'Symbol', symbol)
			r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
			last_record['Insertdate'] = datetime.now().isoformat()
		print(last_record)   
	else:
		print(last_record)   
		print('Không có tín hiệu!')

	# ##############################################Step 4: Vẽ biểu đồ##############################################  
	
	import plotly.graph_objects as go

	# Đảm bảo rằng data['Datetime'] là dạng datetime nếu chưa phải
	# data['Datetime'] = pd.to_datetime(data['Datetime'])

	# Tạo biểu đồ Candlestick cho dữ liệu giá
	fig = go.Figure(data=[go.Candlestick(#x=data['Datetime'],  # Sử dụng cột Datetime thay vì index
										x=data.index,
										open=data['Open'],
										high=data['High'],
										low=data['Low'],
										close=data['Close'],
										name='Candlestick')])

	# Thêm dòng dự đoán giá đóng cửa từ mô hình hồi quy
	fig.add_trace(go.Scatter(x=data.index, y=data['Predicted_Close'], mode='lines', name='Predicted Close'))

	# Thêm điểm mua
	fig.add_trace(go.Scatter(#x=data[data['Buy_Signal']]['Datetime'],
							x=data[data['Buy_Signal']].index,
							y=data[data['Buy_Signal']]['Close'],
							mode='markers',
							marker=dict(color='Green', size=10, symbol='triangle-up'),
							name='Buy Signal'))

	# Thêm điểm bán
	fig.add_trace(go.Scatter(x=data[data['Sell_Signal']].index,
							y=data[data['Sell_Signal']]['Close'],
							mode='markers',
							marker=dict(color='Red', size=10, symbol='triangle-down'),
							name='Sell Signal'))

	# Thêm Momentum và RSI vào trục Y phụ
	fig.add_trace(go.Scatter(x=data.index, y=data['Momentum'], name='Momentum', yaxis='y2'))
	fig.add_trace(go.Scatter(x=data.index, y=data['RSI'], name='RSI', yaxis='y3'))

	# Tùy chỉnh layout để thêm trục Y phụ và cập nhật tiêu đề trục X
	fig.update_layout(
		title=f'Trading Signals for {symbol} based on Linear Regression, Momentum, and RSI',
		xaxis_title='Date',
		yaxis_title='Price',
		xaxis_rangeslider_visible=False,
		yaxis=dict(
			title='Price',
			tickfont=dict(color='#1f77b4')
		),
		yaxis2=dict(
			title='Momentum',
			tickfont=dict(color='#ff7f0e'),
			anchor='free',
			overlaying='y',
			side='left',
			position=0.15
		),
		yaxis3=dict(
			title='RSI',
			tickfont=dict(color='#d62728'),
			anchor='x',
			overlaying='y',
			side='right'
		),
	)

	fig.show()


# Chay Schedule + Timeframe

In [3]:
import MetaTrader5 as mt5
from datetime import datetime, timedelta
import schedule
import time

# Hamh schedule_detectSignal sẽ được gọi theo lịch trình
# Hàm này sẽ được gọi theo lịch trình
def schedule_detectSignal(timeframe):
	symbol = 'GBPUSD.sml' # Symbol tren san giao dich, y chang voi symbol tren san MT5
	from_date = (datetime.now() - timedelta(days=0)).strftime('%Y-%m-%d')  # Lấy ngày hiện tại
	to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
	detectSignal(symbol, from_date, to_date, timeframe)

# Danh sách các phút cụ thể bạn muốn hàm được chạy
run_minutes = list(range(0, 60, 1))  # Gia tri 0,1,2,3,4,5,...59
# Lặp qua từng phút trong danh sách và kiểm tra xem có chạy hàm hay không
while True:
	current_time = datetime.now()
	current_minute = current_time.minute
	current_second = current_time.second

	# Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
	if current_minute in run_minutes and current_second == 0:
		# Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
		last_run = getattr(schedule_detectSignal, 'last_run', None)
		if last_run is None or last_run != current_minute:
			# Gọi hàm detectSignal với timeframe là M1
			schedule_detectSignal(mt5.TIMEFRAME_M1)
			# Lưu lại phút cuối cùng mà hàm đã chạy
			setattr(schedule_detectSignal, 'last_run', current_minute)

	# Nghỉ 1 giây trước khi kiểm tra lại
	time.sleep(1)

'''
# # Danh sách các giờ cụ thể bạn muốn hàm được chạy
# Lưu lại giờ và phút cuối cùng mà hàm đã chạy
last_run_hour = None
last_run_minute = None

while True:
	current_time = datetime.now() # Lấy thời gian hiện tại
	current_hour = current_time.hour # Lấy giờ hiện tại
	current_minute = current_time.minute # Lấy phút hiện tại
	
	# Kiểm tra nếu hàm chưa chạy trong giờ hiện tại
	if current_minute == 0 and (last_run_hour != current_hour or last_run_minute != current_minute):
		schedule_detectSignal(mt5.TIMEFRAME_H1)        
		# Lưu lại giờ và phút cuối cùng mà hàm đã chạy
		last_run_hour = current_hour
		last_run_minute = current_minute
		
	time.sleep(60)  # Đợi 1 phút trước khi kiểm tra lại
'''

                Datetime     Open     High      Low    Close  Volume
0    2025-11-13 17:00:00  1.31812  1.31814  1.31759  1.31768     313
1    2025-11-13 17:01:00  1.31768  1.31773  1.31757  1.31761     332
2    2025-11-13 17:02:00  1.31760  1.31773  1.31758  1.31770     360
3    2025-11-13 17:03:00  1.31770  1.31775  1.31747  1.31749     340
4    2025-11-13 17:04:00  1.31748  1.31781  1.31746  1.31780     290
...                  ...      ...      ...      ...      ...     ...
1361 2025-11-14 15:47:00  1.31640  1.31676  1.31640  1.31665     284
1362 2025-11-14 15:48:00  1.31665  1.31665  1.31635  1.31643     273
1363 2025-11-14 15:49:00  1.31643  1.31647  1.31604  1.31605     313
1364 2025-11-14 15:50:00  1.31606  1.31625  1.31586  1.31618     309
1365 2025-11-14 15:51:00  1.31619  1.31646  1.31602  1.31614     280

[1366 rows x 6 columns]
Mean Squared Error (MSE): 4.932071771503221e-08
                Datetime     Open     High      Low    Close  Volume  \
0    2025-11-13 17:00:00  1

KeyboardInterrupt: 